# Data Pipeline
Covers steps **2.1 → 2.2 → 2.3 → 2.4** of the project.

| Step | Task |
|------|------|
| 2.1  | Εύρεση και Λήψη Dataset (Dynamic Data Ingestion) |
| 2.2  | Καθαρισμός Δεδομένων (Noise removal, NaN handling) |
| 2.3  | Εξαγωγή Χαρακτηριστικών (Feature selection) |
| 2.4  | Διακριτοποίηση (Converting continuous values to symbols) |

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

for candidate in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    if (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None and Path("/content/drive/MyDrive").exists():
    # Fast paths to check instead of slow rglob
    candidate_paths = [
        Path("/content/drive/MyDrive/Entropy-Based Network Intrusion Detection using Markov Chains"),
        Path("/content/drive/MyDrive/IT_PROJECT"),
        Path("/content/drive/MyDrive/IT_Project"),
    ]
    for p in candidate_paths:
        if (p / "config.yaml").exists():
            PROJECT_ROOT = p
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in ["2_Data_Pipeline", "3_Training", "4_Detection", "5_Evaluation"] else NOTEBOOK_DIR

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT verified: {PROJECT_ROOT}")
DATA_PATH = PROJECT_ROOT / "data"


## 2.1 – Εύρεση και Λήψη Dataset (Dynamic Data Ingestion)
We do not load local CSV files. We pull the official CSE-CIC-IDS2018 dataset dynamically from AWS S3 using the `polars` library for high performance.

In [ ]:
import polars as pl

# Direct public AWS S3 URL for the CSE-CIC-IDS2018 dataset (Wednesday 14-02-2018 Traffic)
DATA_URL = "https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv"

print("Downloading and loading data from URL...")
# Load the data directly into a Polars DataFrame
# infer_schema_length=10000 ensures Polars checks enough rows to guess the correct data types
# ignore_errors=True skips any poorly formatted rows during download
df = pl.read_csv(
    DATA_URL,
    infer_schema_length=10000,
    ignore_errors=True
)

print(f"Data loaded successfully! Total rows: {df.height}, Total columns: {df.width}")
df.head()

## 2.2 – Καθαρισμός Δεδομένων (Data Cleaning)
Real-world network data is often messy. We need to fix column names, remove exact duplicates, and drop rows with missing information (`NaN` or `null`).

In [ ]:
# 1. Clean column names (remove leading/trailing spaces)
clean_columns = {col: col.strip() for col in df.columns}
df = df.rename(clean_columns)

# 2. Remove duplicate rows
original_count = df.height
df = df.unique()
print(f"Removed {original_count - df.height} duplicate rows.")

# 3. Remove rows with missing values (nulls)
current_count = df.height
df = df.drop_nulls()
print(f"Removed {current_count - df.height} rows with missing values.")

print(f"Final cleaned dataset shape: {df.height} rows, {df.width} columns")

## 2.3 – Εξαγωγή Χαρακτηριστικών (Feature Selection)
According to **RULE 2** and our literature findings, we must drop redundant features (like IP addresses) and focus on 5 specific behavioral/flow features that strongly detect anomalies like DDoS or port scans.

In [ ]:
# 1. Create 'Out_Degree': count of unique Destination IPs for each Source IP
# We use the .over("Dst IP") or grouped window function in Polars to calculate host out-degree if IP columns exist,
# or fallback to proxy flow groupings if IP fields are anonymized.
if "Dst IP" in df.columns and "Src IP" in df.columns:
    df = df.with_columns(pl.col("Dst IP").n_unique().over("Src IP").alias("Out_Degree"))
else:
    # Fallback for CICFlowMeter CSVs where IPs are omitted: use Dst Port fan-out proxy
    df = df.with_columns(pl.col("Dst Port").n_unique().over("Dst Port").alias("Out_Degree"))

# 2. Synthesize 'TCP_Flags_Int' from individual flag columns
# Formula: SYN + 2*ACK + 4*PSH + 8*RST + 16*URG + 32*FIN
flag_cols = {
    "SYN": "SYN Flag Cnt" if "SYN Flag Cnt" in df.columns else "SYN Flag Count",
    "ACK": "ACK Flag Cnt" if "ACK Flag Cnt" in df.columns else "ACK Flag Count",
    "PSH": "PSH Flag Cnt" if "PSH Flag Cnt" in df.columns else "PSH Flag Count",
    "RST": "RST Flag Cnt" if "RST Flag Cnt" in df.columns else "RST Flag Count",
    "URG": "URG Flag Cnt" if "URG Flag Cnt" in df.columns else "URG Flag Count",
    "FIN": "FIN Flag Cnt" if "FIN Flag Cnt" in df.columns else "FIN Flag Count"
}

df = df.with_columns(
    (
        pl.col(flag_cols["SYN"]) * 1 +
        pl.col(flag_cols["ACK"]) * 2 +
        pl.col(flag_cols["PSH"]) * 4 +
        pl.col(flag_cols["RST"]) * 8 +
        pl.col(flag_cols["URG"]) * 16 +
        pl.col(flag_cols["FIN"]) * 32
    ).alias("TCP_Flags_Int")
)

# 3. Select the 5 critical features defined in config.yaml, plus the Label
flow_dur_col = "Flow Duration" if "Flow Duration" in df.columns else "Flow_Duration"
tot_pkts_col = "Tot Fwd Pkts" if "Tot Fwd Pkts" in df.columns else "Total Fwd Packets"
tot_len_col = "TotLen Fwd Pkts" if "TotLen Fwd Pkts" in df.columns else "Total Length of Fwd Packets"

SELECTED_FEATURES = [
    flow_dur_col,
    "Out_Degree",
    tot_pkts_col,
    tot_len_col,
    "TCP_Flags_Int",
    "Label"
]
df_selected = df.select(SELECTED_FEATURES)

# 4. Rename columns to match config.yaml exactly
df_selected = df_selected.rename({
    flow_dur_col: "Flow_Duration",
    tot_pkts_col: "Tot_Fwd_Pkts",
    tot_len_col: "TotLen_Fwd_Pkts"
})

print(f"Features selected! Dataset shape: {df_selected.shape}")
df_selected.head()

## 2.4 – Διακριτοποίηση (Symbolisation)
Markov chains require discrete "states" (symbols). We convert our continuous numbers into 8 equal-width bins (0 to 7) as specified in `config.yaml`.

In [ ]:
BINS = 8
numeric_cols = ["Flow_Duration", "Out_Degree", "Tot_Fwd_Pkts", "TotLen_Fwd_Pkts"]

# Compress extreme outliers using a log transform so they don't crush the entropy
df_selected = df_selected.with_columns(
    [pl.col(c).clip(lower_bound=0).log1p().alias(f"{c}_log") for c in numeric_cols]
)

# Perform equal-width binning on the logged values
for col_name in numeric_cols:
    col_log = f"{col_name}_log"
    col_min = df_selected.select(pl.col(col_log).min()).item()
    col_max = df_selected.select(pl.col(col_log).max()).item()
    
    if col_max == col_min:
        df_selected = df_selected.with_columns(pl.lit(0).alias(f"{col_name}_sym"))
    else:
        df_selected = df_selected.with_columns(
            ((pl.col(col_log) - col_min) / (col_max - col_min) * (BINS - 1))
            .cast(pl.Int32)
            .alias(f"{col_name}_sym")
        )

# TCP Flags don't need logging, they are already a perfect 64-state integer
df_selected = df_selected.with_columns(
    pl.col("TCP_Flags_Int").cast(pl.Int32).alias("TCP_Flags_Int_sym")
)

df_selected = df_selected.drop([f"{c}_log" for c in numeric_cols])

DATA_SAVE_PATH = DATA_PATH / "processed.parquet"
DATA_SAVE_PATH.parent.mkdir(exist_ok=True)
df_selected.write_parquet(DATA_SAVE_PATH)
print(f"Data successfully discretized with Log-Binning and saved to {DATA_SAVE_PATH}")
df_selected.head()
